In [6]:
import os
from google import genai
import sys
sys.path.append('../..')
import utils

import panel as pn  # GUI
pn.extension()

In [7]:
# Set up Gemini API key

from google import genai

client = genai.Client(api_key="AQ.Ab8RN6KjsbVrgSlYbnOYDCklPfZrv7u9rpjdc4cA3ynLVVhF3w")# Replace with your actual API key

In [8]:

from google import genai

client = genai.Client(api_key="AQ.Ab8RN6KjsbVrgSlYbnOYDCklPfZrv7u9rpjdc4cA3ynLVVhF3w")

models = client.models.list()

print([m.name for m in models])


['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3-pro-image-preview', 'models/gemini-3-pro-image', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.5-transcribe', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/lyria-3-clip-preview', 'models/lyria-3-pro-

In [9]:
from google import genai
from google.genai import types

# Create the Gemini client once
client = genai.Client(
    api_key="AQ.Ab8RN6KjsbVrgSlYbnOYDCklPfZrv7u9rpjdc4cA3ynLVVhF3w"
)


def get_completion_from_messages(
    messages,
    model="gemini-2.5-flash",
    temperature=0.7,
    max_tokens=500
):
    
    # Store system instruction separately
    system_message = ""
    formatted_messages = []

    for message in messages:

        if "content" not in message:
            raise ValueError(
                f"Message missing 'content' field: {message}"
            )

        role = message["role"]
        content = message["content"]

        # Gemini uses "model" instead of "assistant"
        if role == "assistant":
            role = "model"

        # Handle system instruction
        if role == "system":
            system_message = content

        else:
            formatted_messages.append(
                types.Content(
                    role=role,
                    parts=[
                        types.Part(text=content)
                    ]
                )
            )

    # Generate response using the new Gemini API
    response = client.models.generate_content(
        model=model,
        contents=formatted_messages,
        config=types.GenerateContentConfig(
            system_instruction=system_message,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    return response.text

In [10]:
# ============================================
# IMPORTS
# ============================================

from google import genai
from google.genai import types
import utils


# ============================================
# GEMINI CLIENT
# ============================================

client = genai.Client(
    api_key="AQ.Ab8RN6KjsbVrgSlYbnOYDCklPfZrv7u9rpjdc4cA3ynLVVhF3w"
)


# ============================================
# GEMINI COMPLETION FUNCTION
# ============================================

def get_completion_from_messages(
    messages,
    model="gemini-2.5-flash",
    temperature=0.7,
    max_tokens=500
):

    system_message = ""
    formatted_messages = []

    for message in messages:

        if "content" not in message:
            raise ValueError(
                f"Message missing 'content' field: {message}"
            )

        role = message["role"]
        content = message["content"]

        # Convert OpenAI-style assistant role
        # to Gemini's model role
        if role == "assistant":
            role = "model"

        # System instruction
        if role == "system":
            system_message = content

        else:
            formatted_messages.append(
                types.Content(
                    role=role,
                    parts=[
                        types.Part(text=content)
                    ]
                )
            )

    response = client.models.generate_content(
        model=model,
        contents=formatted_messages,
        config=types.GenerateContentConfig(
            system_instruction=system_message,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    return response.text


# ============================================
# PROCESS USER MESSAGE
# ============================================

def process_user_message(
    user_input,
    all_messages,
    debug=True
):

    delimiter = "```"

    # ----------------------------------------
    # STEP 1: INPUT VALIDATION
    # ----------------------------------------

    if debug:
        print("Step 1: Input passed basic check.")

    if not user_input or not user_input.strip():
        raise ValueError("User input cannot be empty.")

    # ----------------------------------------
    # STEP 2: EXTRACT PRODUCTS & CATEGORIES
    # ----------------------------------------

    category_and_product_response = (
        utils.find_category_and_product_only(
            user_input,
            utils.get_products_and_category()
        )
    )

    print("\nCategory/Product response:")
    print(category_and_product_response)

    category_and_product_list = (
        utils.read_string_to_list(
            category_and_product_response
        )
    )

    print("\nCategory/Product list:")
    print(category_and_product_list)

    if debug:
        print(
            f"\nStep 2: Extracted products: "
            f"{category_and_product_list}"
        )

    # ----------------------------------------
    # STEP 3: FETCH PRODUCT INFORMATION
    # ----------------------------------------

    product_information = (
        utils.generate_output_string(
            category_and_product_list
        )
    )

    if debug:
        print(
            "\nStep 3: Retrieved product information."
        )

    # ----------------------------------------
    # STEP 4: GENERATE CUSTOMER RESPONSE
    # ----------------------------------------

    step_4_system_message_content = """
You are a helpful AI customer service assistant.

Answer the customer's question using the relevant
product information provided.

Rules:
- Be accurate and helpful.
- Do not invent product information.
- If information is not available, clearly say so.
- If the customer asks about multiple products,
  answer each product separately.
- Keep the response clear and easy to understand.
"""

    messages = [

        {
            "role": "system",
            "content": step_4_system_message_content
        },

        {
            "role": "user",
            "content": (
                f"Customer message:\n"
                f"{delimiter}\n"
                f"{user_input}\n"
                f"{delimiter}\n\n"
                f"Relevant product information:\n"
                f"{delimiter}\n"
                f"{product_information}\n"
                f"{delimiter}"
            )
        }
    ]

    # Include previous conversation
    final_response = get_completion_from_messages(
        all_messages + messages
    )

    # Save conversation
    all_messages.append(
        {
            "role": "user",
            "content": user_input
        }
    )

    all_messages.append(
        {
            "role": "assistant",
            "content": final_response
        }
    )

    if debug:
        print(
            "\nStep 4: Generated response."
        )

    # ----------------------------------------
    # STEP 5: EVALUATE RESPONSE
    # ----------------------------------------

    step_5_system_message_content = """
You are an AI response quality evaluator.

Determine whether the agent response sufficiently
answers the customer's question.

Reply with ONLY:
Y
or
N

Do not provide any explanation.
"""

    evaluation_messages = [

        {
            "role": "system",
            "content": step_5_system_message_content
        },

        {
            "role": "user",
            "content": (
                f"Customer message:\n"
                f"{delimiter}\n"
                f"{user_input}\n"
                f"{delimiter}\n\n"

                f"Relevant product information:\n"
                f"{delimiter}\n"
                f"{product_information}\n"
                f"{delimiter}\n\n"

                f"Agent response:\n"
                f"{delimiter}\n"
                f"{final_response}\n"
                f"{delimiter}\n\n"

                "Does the response sufficiently answer "
                "the customer's question?\n"
                "Reply ONLY Y or N."
            )
        }
    ]

    evaluation_response = get_completion_from_messages(
        evaluation_messages,
        temperature=0,
        max_tokens=5
    )

    # ----------------------------------------
    # STEP 6: CLEAN EVALUATION
    # ----------------------------------------

    cleaned_evaluation_response = (
        evaluation_response
        .strip()
        .upper()
    )

    # Handle cases such as "Y." or "Y\n"
    if cleaned_evaluation_response.startswith("Y"):
        cleaned_evaluation_response = "Y"

    elif cleaned_evaluation_response.startswith("N"):
        cleaned_evaluation_response = "N"

    if debug:
        print(
            f"\nStep 6: Evaluation result: "
            f"{cleaned_evaluation_response}"
        )

    # ----------------------------------------
    # STEP 7: DECISION
    # ----------------------------------------

    if cleaned_evaluation_response == "Y":

        if debug:
            print(
                "Step 7: Response is approved."
            )

        return final_response, all_messages

    else:

        if debug:
            print(
                "Step 7: Response is not sufficient."
            )

        fallback_response = (
            "I'm unable to provide the information "
            "you're looking for. Let me connect you "
            "with a representative."
        )

        return fallback_response, all_messages


# ============================================
# TEST
# ============================================

user_input = (
    "Tell me about the SmartX Pro Phone and the "
    "FotoSnap Camera, the DSLR one. Also, what "
    "can you tell me about your TVs?"
)

response, all_messages = process_user_message(
    user_input,
    []
)

print("\n========================================")
print("FINAL RESPONSE")
print("========================================")
print(response)

Step 1: Input passed basic check.


InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]

In [ ]:
pn.extension(raw_css=['''
.assistant-response {
    background-color: #F6F6F6;
    padding: 10px;
    border-radius: 5px;
}
'''])

In [9]:
def collect_messages(debug=False):
    user_input = inp.value
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context

    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, css_classes=['assistant-response'])))

    return pn.Column(*panels)

In [10]:
panels = []  # collect display
context = [{'role': 'system', 'content': "You are a Service Assistant"}]

inp = pn.widgets.TextInput(placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Service Assistant')
    [2] ParamFunction(function, _pane=Str, defer_load=False, height=300, loading_indicator=True)